# Work IQ MCP

Work IQ gives agents the signed-in user's Microsoft 365 work context, honoring their permissions and sensitivity labels. Connect through MCP, discover the available tools, ask for an email summary, and then retrieve structured data yourself. This notebook only uses read and discovery tools.

| Tools | Purpose |
| --- | --- |
| `ask`, `list_agents` | Ask natural-language questions and discover agents |
| `search_paths`, `get_schema` | Discover resource paths and their request schemas |
| `fetch` | Read entities |

> In the Skillable lab, Work IQ authentication is already configured. For self-deployment, complete the [Work IQ admin setup](../docs/workiq-admin-setup.md) first. Direct gateway access requires the appropriate Microsoft 365 Copilot entitlement.

## Step 1: Authenticate as the signed-in user

Work IQ calls are delegated, not application-only. Load the app registration from `.env` and acquire a token with MSAL. Sign in with your lab account in the browser when prompted.

In [ ]:
import os

import httpx
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from msal import PublicClientApplication

load_dotenv(override=True)

app_id = os.environ["WORK_IQ_SEARCH_ENTRA_APP_ID"]
tenant_id = os.environ["WORK_IQ_SEARCH_ENTRA_TENANT_ID"]

app = PublicClientApplication(
    client_id=app_id,
    authority=f"https://login.microsoftonline.com/{tenant_id}",
)
auth_result = app.acquire_token_interactive(["api://workiq.svc.cloud.microsoft/WorkIQAgent.Ask"])
if "access_token" not in auth_result:
    raise RuntimeError(f"Authentication failed: {auth_result.get('error_description', auth_result)}")

token = auth_result["access_token"]
print("Signed in to Work IQ")

## Step 2: List the available tools

Open an authenticated HTTP connection, initialize an MCP session, and list the tools before using them. The SDK handles JSON-RPC and streaming responses. The server also exposes write tools; we leave those for the [Work IQ actions notebook](workiq-mcp-actions.ipynb).

In [ ]:
async with httpx.AsyncClient(
    headers={"Authorization": f"Bearer {token}"},
    timeout=httpx.Timeout(30, read=180),
) as http_client:
    async with streamable_http_client("https://workiq.svc.cloud.microsoft/mcp", http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_response = await session.list_tools()

for tool in tools_response.tools:
    print(f"{tool.name}: {tool.description}")
    print("  Required arguments:", tool.inputSchema.get("required", []))

## Step 3: Ask about your work context

`ask` is the natural-language entry point: Work IQ plans the retrieval and returns a synthesized answer with citations. It is an MCP tool, not a separate REST chat route. Ask about the sample emails you explored in Outlook.

In [ ]:
question = "Summarize my recent emails about the Professional Claw Hammer. Who sent them, and what actions do they request?"

async with httpx.AsyncClient(
    headers={"Authorization": f"Bearer {token}"},
    timeout=httpx.Timeout(30, read=180),
) as http_client:
    async with streamable_http_client("https://workiq.svc.cloud.microsoft/mcp", http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("ask", {"question": question})

if result.isError:
    raise RuntimeError(f"Work IQ ask failed: {result.content}")
if result.structuredContent is None:
    raise RuntimeError(f"Work IQ ask returned no structured content: {result.content}")
print(result.structuredContent["answer"])

## Step 4: Discover resource paths

For precise retrieval, discover paths before calling them. `search_paths` accepts a regex filter and returns matching Microsoft 365 resource paths and their supported operations. The model is a small set of verbs over runtime-discovered paths.

In [ ]:
async with httpx.AsyncClient(
    headers={"Authorization": f"Bearer {token}"},
    timeout=httpx.Timeout(30, read=180),
) as http_client:
    async with streamable_http_client("https://workiq.svc.cloud.microsoft/mcp", http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("search_paths", {"filter": "mail"})

if result.isError:
    raise RuntimeError(f"Work IQ search_paths failed: {result.content}")
if result.structuredContent is None:
    raise RuntimeError(f"Work IQ search_paths returned no structured content: {result.content}")
for path in result.structuredContent["paths"][:8]:
    print(path["path"], "->", ", ".join(path["operations"]))

## Step 5: Inspect the email schema

`get_schema` describes a path for a particular operation. Use `operationType="fetch"` to inspect the read schema before fetching emails.

In [ ]:
async with httpx.AsyncClient(
    headers={"Authorization": f"Bearer {token}"},
    timeout=httpx.Timeout(30, read=180),
) as http_client:
    async with streamable_http_client("https://workiq.svc.cloud.microsoft/mcp", http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("get_schema", {"path": "/me/messages", "operationType": "fetch"})

if result.isError:
    raise RuntimeError(f"Work IQ get_schema failed: {result.content}")
for content in result.content:
    if content.type == "text":
        print(content.text)

## Step 6: Fetch structured email data

`fetch` retrieves entities by relative URL. Unlike `ask`, it returns data rather than a synthesized answer. Request five emails with only the fields you want to inspect.

In [ ]:
async with httpx.AsyncClient(
    headers={"Authorization": f"Bearer {token}"},
    timeout=httpx.Timeout(30, read=180),
) as http_client:
    async with streamable_http_client("https://workiq.svc.cloud.microsoft/mcp", http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("fetch", {
                "entityUrls": ["/me/messages?$top=5&$select=subject,from,receivedDateTime"],
            })

if result.isError:
    raise RuntimeError(f"Work IQ fetch failed: {result.content}")
if result.structuredContent is None:
    raise RuntimeError(f"Work IQ fetch returned no structured content: {result.content}")
for response in result.structuredContent["results"]:
    for message in response["data"]["value"][:5]:
        print(message["receivedDateTime"], "-", message["from"], "-", message["subject"])

## Step 7: Discover agents

`list_agents` discovers agents available to the signed-in user. An agent ID can be supplied to `ask` through its optional `agentId` argument.

In [ ]:
import json

async with httpx.AsyncClient(
    headers={"Authorization": f"Bearer {token}"},
    timeout=httpx.Timeout(30, read=180),
) as http_client:
    async with streamable_http_client("https://workiq.svc.cloud.microsoft/mcp", http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("list_agents", {})

if result.isError:
    raise RuntimeError(f"Work IQ list_agents failed: {result.content}")
if result.structuredContent is not None:
    print(json.dumps(result.structuredContent, indent=2))
else:
    for content in result.content:
        if content.type == "text":
            print(json.dumps(json.loads(content.text), indent=2))

## Recap

You connected to Work IQ through MCP, listed tools, asked for a synthesized answer, and used resource paths and schemas to retrieve structured data. Every request ran as the signed-in user, and no write tools were called.

**Optional next steps:** [Work IQ actions](workiq-mcp-actions.ipynb) covers draft, confirm, and write; [Work IQ A2A](workiq-a2a.ipynb) shows task delegation through a different protocol.